# Allen Cell Types — Data Loading & Initial Exploration

**Dataset:** Allen Institute Cell Types Database  
**Reference:** https://alleninstitute.github.io/AllenSDK/cell_types.html

This notebook mirrors `spe-1_load_data.ipynb`. We download raw NWB voltage traces
from Long Square sweeps (constant current injection — direct analog of spe-1's constant-step
patch recordings) and verify they're ready for the spikeparam pipeline.

**Pipeline:**
1. Load cell metadata (species, brain area, cell type, transgenic line)
2. Download NWB files and extract Long Square voltage traces
3. Spot-check a few cells: plot traces, verify spike detection threshold
4. Save population trace dict for downstream spike analysis

## 0. Imports

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Add dataset dir to path so imports work
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

from allen_ct_helper_modules.data_loader import (
    load_cell_metadata,
    get_long_square_sweeps,
    load_voltage_trace,
    load_all_long_square_traces,
    load_population_traces,
    save_to_pickle,
    load_from_pickle,
)
from allen_ct_helper_modules.config import (
    DEFAULT_SPECIES, DEFAULT_STRUCTURES,
    PRIMARY_SPIKE_FEATURES,
    ALLEN_CT_PICKLE_ROOT,
)

## 1. Load Cell Metadata

In [2]:
# Load all mouse cells (default). Change species=None for human+mouse.
cells_df = load_cell_metadata(
    species="Mus musculus",
    structures=None,  # set to e.g. ["VISp"] to restrict to visual cortex
)
print(cells_df.shape)
cells_df.head()

ModuleNotFoundError: No module named 'allensdk'

In [ ]:
# Summary of cell types and brain areas
print("=== Dendrite type ===")
print(cells_df["dendrite_type"].value_counts())
print()
print("=== Brain structure (top 20) ===")
print(cells_df["structure_acronym"].value_counts().head(20))
print()
print("=== Transgenic lines (top 10) ===")
if "transgenic_line" in cells_df.columns:
    print(cells_df["transgenic_line"].value_counts().head(10))

## 2. Spot-Check: Examine Sweeps for a Single Cell

In [ ]:
# Pick a cell to inspect
specimen_id = int(cells_df.iloc[0]["id"])
print(f"Specimen ID: {specimen_id}")
print(cells_df.iloc[0][["structure_acronym", "dendrite_type", "transgenic_line"]])

In [ ]:
# List available Long Square sweeps
sweeps_meta = get_long_square_sweeps(specimen_id)
print(f"{len(sweeps_meta)} Long Square sweeps found")
pd.DataFrame(sweeps_meta)[["sweep_number", "stimulus_name", "stimulus_amplitude"]].head(10)

## 3. Load and Plot Voltage Traces

In [ ]:
# Load one sweep and inspect the voltage trace
# (same check we do in spe-1: plot raw patch voltage to verify it looks right)
voltage, stimulus, times, fs, idx_range = load_voltage_trace(specimen_id, sweeps_meta[5])

print(f"Sampling rate: {fs} Hz")
print(f"Trace length: {len(voltage)/fs:.2f} s ({len(voltage)} samples)")
print(f"Voltage range: {voltage.min():.1f} to {voltage.max():.1f} mV")
print(f"Stimulus amplitude: {sweeps_meta[5].get('stimulus_amplitude', 'N/A')} pA")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

start, stop = idx_range
axes[0].plot(times[start:stop], voltage[start:stop], 'k', linewidth=0.8)
axes[0].set_ylabel("Voltage (mV)")
axes[0].set_title(f"Specimen {specimen_id} — Sweep {sweeps_meta[5]['sweep_number']}")

axes[1].plot(times[start:stop], stimulus[start:stop], 'steelblue', linewidth=0.8)
axes[1].set_ylabel("Stimulus (pA)")
axes[1].set_xlabel("Time (ms)")

plt.tight_layout()
plt.show()

## 4. Verify Spike Detection

Using the same threshold-crossing approach as spe-1's `find_spike_times()`.

In [ ]:
# Trim to recording epoch
start, stop = idx_range
v_trim = voltage[start:stop]
t_trim = times[start:stop]
one_ms = int(fs / 1000)

# Spike detection — same approach as spe-1 (threshold crossing on upswing)
# Adjust thresh_mv as needed for this cell
thresh_mv = -10  # mV — same default as spe-1/pvc-6
thresh_ms_distance = one_ms * 2  # minimum 2 ms between spikes

idx_spikes, props = find_peaks(v_trim, height=thresh_mv, distance=thresh_ms_distance)
amp_spikes = props["peak_heights"]

print(f"Detected {len(idx_spikes)} spikes")
if len(idx_spikes) > 0:
    print(f"Peak amplitudes: {amp_spikes.min():.1f} – {amp_spikes.max():.1f} mV")

In [ ]:
# Plot with detected spikes overlaid
plt.figure(figsize=(14, 4))
plt.plot(t_trim, v_trim, 'k', linewidth=0.8, label='voltage')
plt.plot(t_trim[idx_spikes], v_trim[idx_spikes], 'r.', markersize=8, label='spikes')
plt.axhline(thresh_mv, color='gray', linestyle='--', linewidth=0.8, label=f'threshold ({thresh_mv} mV)')
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.title(f"Specimen {specimen_id} — spike detection")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Inspect Spike Waveforms

Extract windowed waveforms (±10 ms around each spike peak) — same window as spe-1.

In [ ]:
window_pre_ms = 10
window_post_ms = 10
window_pre = int(one_ms * window_pre_ms)
window_post = int(one_ms * window_post_ms)

waveforms = []
for idx in idx_spikes:
    if idx - window_pre >= 0 and idx + window_post < len(v_trim):
        waveforms.append(v_trim[idx - window_pre : idx + window_post])

waveforms = np.array(waveforms)
waveform_times = np.linspace(-window_pre_ms, window_post_ms, waveforms.shape[1])
print(f"Extracted {len(waveforms)} waveforms, shape: {waveforms.shape}")

plt.figure(figsize=(8, 5))
for w in waveforms:
    plt.plot(waveform_times, w, 'k', alpha=0.3, linewidth=0.6)
plt.plot(waveform_times, waveforms.mean(axis=0), 'r', linewidth=2, label='mean')
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel("Time from peak (ms)")
plt.ylabel("Voltage (mV)")
plt.title(f"Specimen {specimen_id} — spike waveforms (n={len(waveforms)})")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Load Population Traces

Download Long Square traces for all (or a subset of) cells.  
⚠️ This downloads NWB files — takes time on first run. Files are cached locally.

In [ ]:
# Start with a small subset to verify pipeline, then increase n_cells
N_CELLS_TEST = 5  # set to None to load all cells

population = load_population_traces(
    cells_df,
    n_cells=N_CELLS_TEST,
    trim_test_pulse=True,
    verbose=True,
)

In [ ]:
# Summary
n_cells_loaded = sum(1 for v in population.values() if len(v) > 0)
n_total_sweeps = sum(len(v) for v in population.values())
print(f"Cells loaded: {n_cells_loaded}")
print(f"Total Long Square sweeps: {n_total_sweeps}")
print(f"Mean sweeps/cell: {n_total_sweeps / max(n_cells_loaded, 1):.1f}")

## 7. Save for Downstream Analysis

In [ ]:
# Save population traces dict — feed into spike waveform analysis notebooks
save_to_pickle(population, "allen_ct_population_traces")

# Save cell metadata
cells_df.to_pickle(os.path.join(ALLEN_CT_PICKLE_ROOT, "allen_ct_cells_df.pkl"))
print("Saved.")

---
**Next step:** `allen_ct_spike_analysis/population_analyses/` — run the same
spikeparam spike waveform clustering pipeline on the extracted traces.